# New-to-Credit Consumer Risk Analysis

Original course project: **Chloe Pham and Grace Ngo**, Dickinson College, May 2025.

This reviewed notebook demonstrates the corrected workflow on **new toy data** because the original coursework CSV was not supplied. Its outputs are software-verification results, not a reproduction of the original experiment. See the README and methodological review for the evidence reconciliation.

In [1]:
from pathlib import Path
import json
import sys

root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'ntc_risk').is_dir()), None)
if root is None:
    raise RuntimeError('Open this notebook from within the project directory.')
sys.path.insert(0, str(root))
from ntc_risk.pipeline import EvaluationConfig, make_demo_data, run_analysis

print('DEMONSTRATION DATA — original coursework CSV is not bundled.')

DEMONSTRATION DATA — original coursework CSV is not bundled.


## Reviewed evaluation protocol

Inputs pass an explicit schema and feature allowlist. The sample is split before train-only imputation. Validation average precision selects the model; a predeclared recall floor and minimum flag count constrain threshold selection. The selected model and threshold are frozen before the test evaluation. All results below use newly generated toy profiles.

In [2]:
demo = make_demo_data(n=600, seed=42)
report, comparison = run_analysis(
    demo, root / 'outputs' / 'notebook-demo', dataset_kind='demonstration',
    config=EvaluationConfig(seed=42),
)
print(report['status'])
print(comparison[['model', 'average_precision', 'threshold', 'precision', 'recall']].to_string(index=False))
print('Frozen selection:', report['selected_model'], report['selected_threshold'])
print('Demo test metrics:', json.dumps(report['selected_model_test'], indent=2))

Demonstration only; not the original project results.
              model  average_precision  threshold  precision   recall
logistic_regression           0.731627       0.70   0.666667 0.526316
      random_forest           0.585952       0.40   0.520000 0.684211
            xgboost           0.472811       0.25   0.384615 0.526316
        dummy_prior           0.211111       0.05   0.211111 1.000000
Frozen selection: logistic_regression 0.7
Demo test metrics: {
  "threshold": 0.7,
  "n": 90,
  "prevalence": 0.2,
  "roc_auc": 0.816358024691358,
  "average_precision": 0.5509613537630779,
  "brier_score": 0.1840283540813104,
  "accuracy": 0.8111111111111111,
  "precision": 0.5263157894736842,
  "recall": 0.5555555555555556,
  "flag_rate": 0.2111111111111111,
  "tn": 63,
  "fp": 9,
  "fn": 8,
  "tp": 10
}


## Interpretation

The ranking above applies only to this generated verification dataset. It does not establish a new result for the original project. Positive flags indicate a retrospective model classification, not a lending action. Scores remain uncalibrated, and no financial benefit or fairness conclusion is inferred.

## Historical evidence

The following records come from saved text in the original notebook. Their feature and preprocessing issues are documented in `docs/methodology_review.md`. In particular, the original Random Forest test row uses a different classification rule from its validation row. These rows are not outputs from the corrected workflow.

In [3]:
historical = json.loads((root / 'docs' / 'historical_results.json').read_text())
print(historical['status'])
for row in historical['records']:
    print(row['model'], row['split'], 'threshold=', row['threshold'],
          'precision=', round(row['precision'], 4), 'recall=', round(row['recall'], 4),
          'saved AUC=', row['roc_auc_reported'])

Saved original notebook outputs; not rerun or endorsed as unbiased performance.
random_forest validation threshold= 0.7 precision= 0.6667 recall= 0.6 saved AUC= 0.953
random_forest test threshold= 0.5 precision= 0.5323 recall= 1.0 saved AUC= 0.943
xgboost validation threshold= 0.7 precision= 0.5455 recall= 0.6 saved AUC= 0.944
xgboost test threshold= 0.7 precision= 0.5882 recall= 0.6061 saved AUC= 0.935


## Reproduce with the original data

Supply an authorized CSV locally, then run `python -m ntc_risk --data data/v_credit_data_NTC_v51825.csv --output outputs/coursework`. Review feature timing and the meaning of `Ever_ChargeOff` before interpreting a run as anything beyond historical classification. Raw records and local exports are excluded from Git.